<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/06_six_hallmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gseapy
from google.colab import drive
drive.mount('/content/drive')

import json, pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/longevity_project'
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')
senmayo = json.load(open(f'{BASE}/data/processed/senmayo_genes.json'))

bing = genes[genes['pr_is_bing'] == 1]
id_to_symbol = dict(zip(bing['pr_gene_id'].astype(str), bing['pr_gene_symbol']))
symbols = set(id_to_symbol.values())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 10.9 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
from gseapy import Msigdb
msig = Msigdb()
DBVER = '2023.2.Hs'   # use the same version that worked in Session 3
h = msig.get_gmt(category='h.all', dbver=DBVER)
bp = msig.get_gmt(category='c5.go.bp', dbver=DBVER)

hallmarks = {
    'senescence':       {'source': 'SAUL_SEN_MAYO', 'genes': senmayo, 'good': 'down'},
    'nutrient_sensing': {'source': 'HALLMARK_MTORC1_SIGNALING', 'genes': h['HALLMARK_MTORC1_SIGNALING'], 'good': 'down'},
    'mitochondria':     {'source': 'HALLMARK_OXIDATIVE_PHOSPHORYLATION', 'genes': h['HALLMARK_OXIDATIVE_PHOSPHORYLATION'], 'good': 'up'},
    'autophagy':        {'source': 'GOBP_MACROAUTOPHAGY', 'genes': bp['GOBP_MACROAUTOPHAGY'], 'good': 'up'},
    'inflammation':     {'source': 'HALLMARK_INFLAMMATORY_RESPONSE', 'genes': h['HALLMARK_INFLAMMATORY_RESPONSE'], 'good': 'down'},
    'proteostasis':     {'source': 'HALLMARK_UNFOLDED_PROTEIN_RESPONSE', 'genes': h['HALLMARK_UNFOLDED_PROTEIN_RESPONSE'], 'good': 'down'},
}
for name, hm in hallmarks.items():
    found = [g for g in hm['genes'] if g in symbols]
    print(f"{name:17} good={hm['good']:4} {len(found)} of {len(hm['genes'])} genes in data")

json.dump(hallmarks, open(f'{BASE}/data/processed/hallmarks.json', 'w'), indent=2)

senescence        good=down 106 of 125 genes in data
nutrient_sensing  good=down 192 of 200 genes in data
mitochondria      good=up   179 of 200 genes in data
autophagy         good=up   237 of 335 genes in data
inflammation      good=down 171 of 200 genes in data
proteostasis      good=down 105 of 113 genes in data


In [3]:
pilot = pd.read_csv(f'{BASE}/data/processed/senescence_pilot_drugs.csv')
named = vcap[~vcap['pert_iname'].str.startswith('BRD-')].copy()
named['name'] = named['pert_iname'].str.lower()

stress = ['tunicamycin', 'thapsigargin', 'bortezomib', 'geldanamycin', 'tanespimycin', 'mg-132']
stress_here = [d for d in stress if d in set(named['name'])]
extra = named[named['name'].isin(stress_here)].copy()
extra['is_24h'] = extra['pert_time'] == 24
extra = extra.sort_values(['is_24h', 'pert_dose'], ascending=False).groupby('name').head(1)

test = pd.concat([pilot, extra]).drop_duplicates('name')
print('Stress drugs found:', stress_here, '| Total drugs:', len(test))

Stress drugs found: ['tunicamycin', 'thapsigargin', 'bortezomib', 'geldanamycin', 'tanespimycin', 'mg-132'] | Total drugs: 106


In [4]:
sigs = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet', columns=list(id_to_symbol))
sigs = sigs.loc[test['sig_id']]
sigs.columns = [id_to_symbol[c] for c in sigs.columns]
print(sigs.shape)

(106, 10174)


In [5]:
import gseapy as gp
from tqdm import tqdm

gene_sets = {name: hm['genes'] for name, hm in hallmarks.items()}
rows = []
for sig_id, row in tqdm(sigs.iterrows(), total=len(sigs)):
    res = gp.prerank(rnk=row.sort_values(ascending=False), gene_sets=gene_sets,
                     permutation_num=100, min_size=5, max_size=1000,
                     seed=0, outdir=None, verbose=False)
    nes = res.res2d.set_index('Term')['NES'].astype(float)
    rows.append({'sig_id': sig_id, **nes.to_dict()})

scores = pd.DataFrame(rows).set_index('sig_id')
for name, hm in hallmarks.items():
    if hm['good'] == 'down':
        scores[name] = -scores[name]
scores = scores.join(test.set_index('sig_id')['name']).set_index('name')
scores.round(3).to_csv(f'{BASE}/results/six_hallmark_pilot.csv')
print(scores.shape)

  0%|          | 0/106 [00:00<?, ?it/s]2026-10-06 18:52:47,953 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  1%|          | 1/106 [00:00<00:11,  9.02it/s]2026-10-06 18:52:48,048 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-10-06 18:52:48,141 [WARNING] Duplicated values found in preranked stats: 0.04% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  3%|▎         | 3/106 [00:00<00:09, 10.34it/s]2026-10-06 18:52:48,231 [WARNING] Duplicated values found in preranked stats: 5.52% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-10-06 18:52:48,328 [WARNING] Duplicated values found in preranked stats: 0.06% of genes
The order of those genes will be arbitrary, which may produce unexpected r

(106, 6)


In [6]:
ranks = scores.rank(ascending=False).astype(int)
check = [d for d in ['sirolimus', 'everolimus'] + stress_here if d in scores.index]
print('Scores:\n', scores.loc[check].round(2))
print(f'\nRanks (1 = most helpful, out of {len(scores)}):\n', ranks.loc[check])

Scores:
               proteostasis  senescence  inflammation  mitochondria  \
name                                                                 
sirolimus             2.15       -0.60          0.81         -0.97   
everolimus            1.07       -1.16         -1.55         -2.11   
tunicamycin          -2.39       -0.73         -1.59         -2.55   
thapsigargin          1.20       -1.02         -0.92         -1.81   
bortezomib           -1.61       -1.65         -1.97         -2.26   
geldanamycin         -1.92       -1.92         -2.13         -2.32   
tanespimycin          1.44        0.90         -1.17         -1.61   
mg-132               -1.18       -1.93         -2.35         -2.96   

              nutrient_sensing  autophagy  
name                                       
sirolimus                 2.51       1.69  
everolimus                0.90       1.63  
tunicamycin              -1.45       1.37  
thapsigargin              2.10      -1.00  
bortezomib               -

In [7]:
n = len(scores)
print(f"X: sirolimus ranked {ranks.loc['sirolimus', 'nutrient_sensing']} of {n} on nutrient sensing")
print('Y: stress drugs on proteostasis:',
      ', '.join(f"{d} {ranks.loc[d, 'proteostasis']}" for d in stress_here), f'(of {n})')

X: sirolimus ranked 3 of 106 on nutrient sensing
Y: stress drugs on proteostasis: tunicamycin 106, thapsigargin 34, bortezomib 84, geldanamycin 99, tanespimycin 23, mg-132 61 (of 106)
